# Классификация SI > 8



In [1]:
# Установка библиотек для Google Colab
!pip -q install xgboost lightgbm openpyxl seaborn


In [2]:
# Загрузка исходного файла с данными

import os
from google.colab import files

DATA_PATH = '/content/drug_data.xlsx'

if not os.path.exists(DATA_PATH):
    uploaded = files.upload()
    if 'drug_data.xlsx' not in uploaded:

        first_file = next(iter(uploaded.keys()))
        os.rename(first_file, DATA_PATH)

os.makedirs('reports', exist_ok=True)
os.makedirs('models', exist_ok=True)
print(f'Файл данных: {DATA_PATH}')
print('Папки reports/ и models/ готовы')


Файл данных: /content/drug_data.xlsx
Папки reports/ и models/ готовы


## Основной код

In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import *
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
import lightgbm as lgb
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer # Import SimpleImputer
import warnings
warnings.filterwarnings('ignore')

class SIHighSelectivityPredictor:
    def __init__(self):
        self.models = {}
        self.results = {}
        self.threshold = 8.0

    def load_and_prepare_data(self):
        print("="*60)
        print("КЛАССИФИКАЦИЯ: SI > 8 (ВЫСОКАЯ СЕЛЕКТИВНОСТЬ)")
        print("="*60)

        df = pd.read_excel(DATA_PATH)
        df = df.rename(columns={'IC50, mM': 'IC50', 'CC50, mM': 'CC50'})

        if 'Unnamed: 0' in df.columns:
            df = df.drop('Unnamed: 0', axis=1)

        print(f"Порог SI: {self.threshold}")

        y = (df['SI'] > self.threshold).astype(int)
        print(f"SI <= 8 (0): {(y == 0).sum()} ({(y == 0).mean()*100:.1f}%)")
        print(f"SI > 8 (1):  {(y == 1).sum()} ({(y == 1).mean()*100:.1f}%)")

        # Проверяем дисбаланс классов
        if (y == 1).mean() < 0.1:
            print("⚠️ Сильный дисбаланс классов - менее 10% положительных примеров")

        feature_cols = [col for col in df.columns if col not in ['IC50', 'CC50', 'SI']]
        X = df[feature_cols].copy()

        # Очистка данных
        constant_features = X.columns[X.nunique() <= 1]
        if len(constant_features) > 0:
            X = X.drop(constant_features, axis=1)

        # Handle NaN values by imputing with the mean
        imputer = SimpleImputer(strategy='mean')
        X = pd.DataFrame(imputer.fit_transform(X), columns=X.columns, index=X.index)

        self.X_train, self.X_test, self.y_train, self.y_test = train_test_split(
            X, y, test_size=0.2, random_state=42, stratify=y
        )

        print(f"Итоговое количество признаков: {X.shape[1]}")
        print(f"Обучающая выборка - класс 1: {(self.y_train == 1).sum()}/{len(self.y_train)} ({(self.y_train == 1).mean()*100:.1f}%)者に)")

        return X, y

    def define_models(self):
        # Учитываем дисбаланс классов
        self.models = {
            'Logistic Regression': Pipeline([
                ('scaler', StandardScaler()),
                ('model', LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced'))
            ]),
            'Random Forest': RandomForestClassifier(
                n_estimators=100, random_state=42, n_jobs=-1, class_weight='balanced'
            ),
            'XGBoost': xgb.XGBClassifier(
                random_state=42, n_jobs=-1, verbosity=0, eval_metric='logloss',
                scale_pos_weight=3  # Компенсация дисбаланса
            ),
            'LightGBM': lgb.LGBMClassifier(
                random_state=42, n_jobs=-1, verbosity=-1, class_weight='balanced'
            )
        }

    def evaluate_and_save(self):
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        final_results = {}

        print("\nОценка моделей:")
        print("-"*40)

        for name, model in self.models.items():
            # Кросс-валидация с несколькими метриками
            f1_scores = cross_val_score(model, self.X_train, self.y_train, cv=cv, scoring='f1')
            auc_scores = cross_val_score(model, self.X_train, self.y_train, cv=cv, scoring='roc_auc')

            print(f"{name}:")
            print(f"  F1:  {f1_scores.mean():.3f} ± {f1_scores.std():.3f}")
            print(f"  AUC: {auc_scores.mean():.3f} ± {auc_scores.std():.3f}")

            # Обучение и тестирование
            model.fit(self.X_train, self.y_train)
            y_pred = model.predict(self.X_test)
            y_pred_proba = model.predict_proba(self.X_test)[:, 1]

            final_results[name] = {
                'accuracy': accuracy_score(self.y_test, y_pred),
                'precision': precision_score(self.y_test, y_pred, zero_division=0),
                'recall': recall_score(self.y_test, y_pred, zero_division=0),
                'f1': f1_score(self.y_test, y_pred, zero_division=0),
                'auc': roc_auc_score(self.y_test, y_pred_proba) if len(np.unique(self.y_test)) > 1 else 0,
                'y_pred': y_pred,
                'y_pred_proba': y_pred_proba
            }

            print(f"  Тест - Precision: {final_results[name]['precision']:.3f}, Recall: {final_results[name]['recall']:.3f}")

        # Сохранение результатов
        results_summary = []
        for name, metrics in final_results.items():
            results_summary.append({
                'Model': name,
                'Accuracy': metrics['accuracy'],
                'Precision': metrics['precision'],
                'Recall': metrics['recall'],
                'F1_Score': metrics['f1'],
                'ROC_AUC': metrics['auc']
            })

        results_df = pd.DataFrame(results_summary)
        results_df.to_csv('reports/si_high_selectivity_classification_results.csv', index=False)

        # Лучшая модель по F1-score
        best_model_name = max(final_results.keys(), key=lambda x: final_results[x]['f1'])

        print(f"\n{'='*40}")
        print(f"ЛУЧШАЯ МОДЕЛЬ: {best_model_name}")
        print(f"F1-Score: {final_results[best_model_name]['f1']:.3f}")
        print(f"Precision: {final_results[best_model_name]['precision']:.3f}")
        print(f"Recall: {final_results[best_model_name]['recall']:.3f}")
        print(f"AUC: {final_results[best_model_name]['auc']:.3f}")

        # Сохранение предсказаний
        predictions_df = pd.DataFrame({
            'True_Class': self.y_test.values,
            'Predicted_Class': final_results[best_model_name]['y_pred'],
            'Predicted_Probability': final_results[best_model_name]['y_pred_proba'],
            'SI_Threshold': self.threshold
        })
        predictions_df.to_csv('reports/si_high_selectivity_predictions.csv', index=False)

        # Анализ результатов
        tn, fp, fn, tp = confusion_matrix(self.y_test, final_results[best_model_name]['y_pred']).ravel()
        print(f"\nМатрица ошибок (лучшая модель):")
        print(f"True Negatives: {tn}")
        print(f"False Positives: {fp}")
        print(f"False Negatives: {fn}")
        print(f"True Positives: {tp}")

        return final_results, best_model_name

def main():
    predictor = SIHighSelectivityPredictor()
    X, y = predictor.load_and_prepare_data()
    predictor.define_models()
    final_results, best_model_name = predictor.evaluate_and_save()

    print("\nКлассификация SI > 8 завершена!")
    print("Результаты сохранены в папку reports/")

if __name__ == "__main__":
    main()


КЛАССИФИКАЦИЯ: SI > 8 (ВЫСОКАЯ СЕЛЕКТИВНОСТЬ)
Порог SI: 8.0
SI <= 8 (0): 644 (64.3%)
SI > 8 (1):  357 (35.7%)
Итоговое количество признаков: 192
Обучающая выборка - класс 1: 285/800 (35.6%)者に)

Оценка моделей:
----------------------------------------
Logistic Regression:
  F1:  0.556 ± 0.039
  AUC: 0.697 ± 0.021
  Тест - Precision: 0.483, Recall: 0.597
Random Forest:
  F1:  0.564 ± 0.020
  AUC: 0.726 ± 0.023
  Тест - Precision: 0.597, Recall: 0.514
XGBoost:
  F1:  0.573 ± 0.023
  AUC: 0.717 ± 0.018
  Тест - Precision: 0.583, Recall: 0.583
LightGBM:
  F1:  0.569 ± 0.045
  AUC: 0.721 ± 0.023
  Тест - Precision: 0.569, Recall: 0.569

ЛУЧШАЯ МОДЕЛЬ: XGBoost
F1-Score: 0.583
Precision: 0.583
Recall: 0.583
AUC: 0.725

Матрица ошибок (лучшая модель):
True Negatives: 99
False Positives: 30
False Negatives: 30
True Positives: 42

Классификация SI > 8 завершена!
Результаты сохранены в папку reports/
